# 1. Feature extraction data cleaning (7-day windows)

7-day-window version of `../MrP_pipeline/1.feature_exaction_data_cleaning.ipynb`; the old notebook is kept as-is for the biweekly results.

**Input**
- `feature_extraction/all_results_7d.csv` -- raw output of `restart_feature_extraction_pipeline/2.extract_belief_features.ipynb` (one row per user x 7-day window)
- `updated_belief_features.json` -- the 40 belief / 10 demo questions and their option lists
- `tl_2025_us_state/tl_2025_us_state.shp` -- state order for `state_cleaned_idx`

**Output**
- `feature_extraction/cleaned_all_result_7d.pkl` / `.csv` -- all rows, beliefs recoded with `_Symbol_num` / `_Symbol_cat`, demographics renamed, `state_cleaned` added
- `feature_extraction/preprocessed_cleaned_all_result_7d.pkl` / `.csv` -- MrP input: `USA` / `No_state` / under-18 rows dropped, predictor `_idx` columns added
- `cleaned_MrP_data/all_mappings_7d.pkl` -- how each predictor was mapped to its index

**Differences from the biweekly notebook**
- New symbols are already plain option positions (`1`, `2`, ...), so the old symbol-regex / prefix-unifying / binary-collapsing / fossil-fuel-composite steps are gone -- the new questions already match the survey codebooks.
- Belief recodes (DIVLAW, COURTS, V241409x) come from `belief_recodes_7d.py`, shared with `18.build_target_df_7d.ipynb` so targets get the same recode.
- Demographic `_idx` = option position (age: position - 1 after dropping under 18) instead of `LabelEncoder`, which renumbered whenever an option happened to be absent. The option order matches `2.stratification_frame_cleaning.ipynb`'s encoding exactly -- checked in section 4.
- Demographics stay per window, as before.

In [ ]:
import json
import pickle

import geopandas as gpd
import numpy as np
import pandas as pd

from belief_recodes_7d import RECODES, n_categories, recode_value

DATA_DIR = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data"
INPUT_CSV = f"{DATA_DIR}/feature_extraction/all_results_7d.csv"
FEATURES_JSON = f"{DATA_DIR}/updated_belief_features.json"
SHAPEFILE = f"{DATA_DIR}/tl_2025_us_state/tl_2025_us_state.shp"

CLEANED_OUT = f"{DATA_DIR}/feature_extraction/cleaned_all_result_7d"
PREPROCESSED_OUT = f"{DATA_DIR}/feature_extraction/preprocessed_cleaned_all_result_7d"
MAPPINGS_OUT = f"{DATA_DIR}/cleaned_MrP_data/all_mappings_7d.pkl"

## 1. Load

Only the columns needed downstream -- the tweet text, raw JSON replies and reasoning traces stay in `all_results_7d.csv`.

In [ ]:
HEAVY_COLS = {
    "tweets_struct", "Belief_Raw", "Demo_Raw", "Belief_Reasoning", "Demo_Reasoning",
    "description", "profile_image_url", "profile_image_description", "profile_image_main_info", "name",
}
df = pd.read_csv(INPUT_CSV, usecols=lambda c: c not in HEAVY_COLS, encoding="utf-8-sig", low_memory=False)

with open(FEATURES_JSON, "r", encoding="utf-8") as f:
    feature_data = json.load(f)
belief_features = feature_data["beliefs"]
demo_features = feature_data["demo"]

print(f"{len(df)} rows, {df['id'].nunique()} users, {(~df['is_empty_window']).sum()} non-empty windows")
print(f"{len(belief_features)} belief / {len(demo_features)} demo questions")

In [ ]:
# every question in the JSON should have its three answer columns
missing_cols = [
    f"{prefix}_{feature['id']}_{part}"
    for prefix, features in [("Belief", belief_features), ("Demo", demo_features)]
    for feature in features
    for part in ["Category", "Symbol", "Speculation"]
    if f"{prefix}_{feature['id']}_{part}" not in df.columns
]
assert not missing_cols, missing_cols

# symbols come back from the CSV as floats (NaN for empty / skipped windows) -- make them nullable ints
symbol_cols = [c for c in df.columns if c.endswith("_Symbol") and c.startswith(("Belief_", "Demo_"))]
df[symbol_cols] = df[symbol_cols].astype("Int64")

## 2. Skipped calls

Empty windows have no answers at all. Non-empty windows whose belief or demo call was skipped (`context_length_exceeded` / `exhausted_retries`) have no answers for that side -- confirm that, so later `dropna` handles both cases the same way.

In [ ]:
nonempty = ~df["is_empty_window"]
for prefix, features in [("Belief", belief_features), ("Demo", demo_features)]:
    skipped = df[f"{prefix}_Skip_Reason"].notna()
    cols = [f"{prefix}_{feature['id']}_Symbol" for feature in features]
    assert df.loc[skipped, cols].isna().all().all(), f"{prefix}: skipped rows have answers"
    assert df.loc[nonempty & ~skipped, cols].notna().all().all(), f"{prefix}: answered rows have missing answers"
    reasons = df.loc[skipped, f"{prefix}_Skip_Reason"].str.split(":").str[0].value_counts().to_dict()
    print(f"{prefix}: {skipped.sum()} skipped non-empty windows {reasons}")

## 3. Beliefs

`Belief_{id}_Symbol_num` = recoded option position (1..K), `Belief_{id}_Symbol_cat` = the same as an ordered categorical with the full 1..K range (`21.2.multi_belief_pipelineV2.py` takes K from the categories, and the Stan model needs 1..K even if a category never appears). `_Symbol` itself is left as the raw position the model chose.

In [ ]:
belief_cols = []
belief_info = []
for feature in belief_features:
    key = feature["id"]
    col = f"Belief_{key}_Symbol"
    n_options = len(feature["options"])
    k = n_categories(key, n_options)

    recoded = df[col].map(lambda v: recode_value(key, v), na_action="ignore").astype("Int64")
    # the Stan model needs y in 1..K with no gaps
    assert sorted({recode_value(key, i) for i in range(1, n_options + 1)}) == list(range(1, k + 1)), key

    df[f"{col}_num"] = recoded
    df[f"{col}_cat"] = pd.Categorical(recoded, categories=list(range(1, k + 1)), ordered=True)
    belief_cols.append(col)
    belief_info.append({"id": key, "column": col, "n_options": n_options, "K": k,
                        "recoded": key in RECODES, "question": feature["feature_name"]})

belief_info = pd.DataFrame(belief_info)
print(f"{len(belief_cols)} belief columns, recoded: {belief_info.loc[belief_info['recoded'], 'id'].tolist()}")
belief_info[["id", "n_options", "K", "recoded"]]

In [ ]:
# before / after for the recoded beliefs (share of answered windows)
for key in RECODES:
    col = f"Belief_{key}_Symbol"
    before = df[col].value_counts(normalize=True).sort_index().round(3).to_dict()
    after = df[f"{col}_num"].value_counts(normalize=True).sort_index().round(3).to_dict()
    print(f"{key}\n  before: {before}\n  after:  {after}")

## 4. Demographics

Rename the six MrP predictors to the names `21.2.multi_belief_pipelineV2.py` expects, and map STATE to abbreviations. The other demo questions (I, M, Pid) keep their ids.

In [ ]:
DEMO_RENAME = {"S": "sex", "A": "age", "E": "ethnicity", "H": "income", "Q": "education", "Vpa": "past_vote"}

df = df.rename(columns={
    f"Demo_{key}_{part}": f"Demo_{name}_{part}"
    for key, name in DEMO_RENAME.items()
    for part in ["Category", "Symbol", "Speculation"]
})
demo_options = {DEMO_RENAME.get(f["id"], f["id"]): f["options"] for f in demo_features}

In [ ]:
STATE_ABBR = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
    # the two fallback options -- no poststratification cell, dropped in section 6
    "USA": "USA",
    "Not from a state in the USA": "No_state",
}

# STATE is now a fixed-choice question, so every answer should be one of its options
assert set(demo_options["STATE"]) == set(STATE_ABBR), set(demo_options["STATE"]) ^ set(STATE_ABBR)
df["state_cleaned"] = df["Demo_STATE_Category"].map(STATE_ABBR)
assert df.loc[df["Demo_STATE_Category"].notna(), "state_cleaned"].notna().all()

df["state_cleaned"].value_counts(dropna=False).head(12)

## 5. Save cleaned data

In [ ]:
df.to_pickle(f"{CLEANED_OUT}.pkl")
df.to_csv(f"{CLEANED_OUT}.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {CLEANED_OUT}.pkl / .csv")

## 6. MrP preprocessing

Drop rows whose state or age has no poststratification cell (`USA`, `No_state`, under 18), then add the predictor indices. Rows with no demo answers (empty or skipped windows) are kept here, like the biweekly notebook did -- `21.2.multi_belief_pipelineV2.py` drops them with its own `dropna`.

In [ ]:
n_before = len(df)
mrp = df[~df["state_cleaned"].isin(["USA", "No_state"])].copy()
n_after_state = len(mrp)
mrp = mrp[mrp["Demo_age_Symbol"].ne(1).fillna(True)].copy()  # option 1 = "under 18 years old"; keeps rows with no age answer
print(f"{n_before} rows -> {n_after_state} after dropping USA / No_state "
      f"-> {len(mrp)} after dropping under 18")

In [ ]:
# Encoding used by 2.stratification_frame_cleaning.ipynb -- copied here so the index each
# option gets can be checked against it, label by label, in the next cell.
SF_LABELS = {
    "sex": ["M", "F"],
    "age": ["1.18-24", "2.25-34", "3.35-44", "4.45-54", "5.55-64", "6.65+"],
    "ethnicity": ["White", "Black", "Hispanic", "Asian", "Native", "Mixed-race"],
    "income": ["1.[min, 25000)", "2.[25000, 50000)", "3.[50000, 75000)", "4.[75000, 100000)", "5.[100000, max]"],
    "education": ["Does not have a College Degree", "Has a College Degree"],
    "past_vote": ["stay home", "R", "D", "L", "G", "OTHER"],
}
AGE_OFFSET = 1  # age option 1 is "under 18", which has no cell, so position 2 -> idx 1

all_mappings = {}
mapping_tables = []
for name in DEMO_RENAME.values():
    offset = AGE_OFFSET if name == "age" else 0
    options = demo_options[name][offset:]
    assert len(options) == len(SF_LABELS[name]), name

    mrp[f"Demo_{name}_idx"] = mrp[f"Demo_{name}_Symbol"] - offset
    observed = mrp[f"Demo_{name}_idx"].dropna()
    assert observed.between(1, len(options)).all(), name

    all_mappings[f"Demo_{name}_Symbol"] = {i + 1 + offset: i + 1 for i in range(len(options))}
    mapping_tables.append(pd.DataFrame({
        "predictor": name, "idx": range(1, len(options) + 1),
        "our_option": [o[:60] for o in options], "strat_frame_label": SF_LABELS[name],
    }))

# eyeball check: each row's option should mean the same as the strat-frame label next to it
pd.set_option("display.max_colwidth", 70)
pd.concat(mapping_tables, ignore_index=True)

In [ ]:
# state index: shapefile order without territories, same as 2.stratification_frame_cleaning.ipynb
shp = gpd.read_file(SHAPEFILE)
shp["STATE"] = shp["STUSPS"].str.upper()
shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])].sort_values("STATE").reset_index(drop=True)
state_to_idx = {s: i + 1 for i, s in enumerate(shp["STATE"])}
assert len(state_to_idx) == 51

mrp["state_cleaned_idx"] = mrp["state_cleaned"].map(state_to_idx)
assert mrp.loc[mrp["state_cleaned"].notna(), "state_cleaned_idx"].notna().all()
all_mappings["state_cleaned"] = state_to_idx

print(f"{mrp['state_cleaned'].nunique()} states present")

## 7. Save MrP input

In [ ]:
mrp.to_pickle(f"{PREPROCESSED_OUT}.pkl")
mrp.to_csv(f"{PREPROCESSED_OUT}.csv", index=False, encoding="utf-8-sig")
with open(MAPPINGS_OUT, "wb") as f:
    pickle.dump(all_mappings, f)

predictor_cols = [f"Demo_{name}_idx" for name in DEMO_RENAME.values()] + ["state_cleaned_idx"]
usable = mrp.dropna(subset=predictor_cols)
print(f"Saved {len(mrp)} rows to {PREPROCESSED_OUT}.pkl / .csv")
print(f"  {len(usable)} rows with all predictors, {usable['id'].nunique()} users, "
      f"{usable['time_window_start'].nunique()} time windows")
print(f"Saved mappings to {MAPPINGS_OUT}")